# Notebook 1 of 5: Data loading, checks and exploration
### Swahili Tweet Sentiment Classification (AfriSenti)

**Dataset:** AfriSenti (Muhammad et al., 2023), Swahili part: tweets labelled **positive, neutral or negative** by native speakers. It comes with an official training, development and test set from the SemEval-2023 Task 12 shared task. The files are plain TSV files on GitHub, so no account or approval is needed.

This notebook:
1. Sets up the project folder
2. Downloads the three Swahili files and checks them
3. Explores class balance, tweet length, vocabulary and a few real examples
4. **Checks for duplicate tweets and overlap between train, validation and test** (an important check, because overlap inflates scores)
5. Cleans the text and saves the fixed split

**Split used by every model:** train = official training file, validation = official development file, test = official test file. Nothing is re-split, so your results can be compared with the shared task's.

## Step 0 - Set up the project folder

Pick the `.venv` kernel (top right of the notebook). The libraries come from `requirements.txt` (`pip install -r requirements.txt` in the `.venv`).

The next cell moves to the project root. Everything this notebook writes goes inside the project folder, so there is nothing to clone or upload.

In [ ]:
from getpass import getpass
from pathlib import Path
import os

# VS Code starts a notebook inside its own folder (notebooks/). Move to the project root so the
# relative paths used below (data/, figures/, results/ ...) resolve from there.
ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "notebooks").is_dir(), "Open the project folder in VS Code and run this notebook from inside it."
os.chdir(ROOT)
print("Project folder:", ROOT)

for folder in ["data/raw", "data/processed", "notebooks", "figures/eda", "results/metrics", "results/confusion_matrices", "results/errors"]:
    os.makedirs(folder, exist_ok=True)

## Step 1 - Download the three Swahili files

Source: the shared task's GitHub repository (`afrisenti-semeval/afrisent-semeval-2023`, Subtask A, Swahili). The licence is CC BY 4.0.

In [ ]:
import urllib.request, os

BASE = "https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/SubtaskA"
FILES = {"train": "train/sw_train.tsv", "val": "dev/sw_dev.tsv", "test": "test/sw_test.tsv"}

for split, path in FILES.items():
    dest = f"data/raw/sw_{split}.tsv"
    if not os.path.exists(dest):
        urllib.request.urlretrieve(f"{BASE}/{path}", dest)
    print(split, "->", dest, f"{os.path.getsize(dest)/1e3:.0f} KB")

## Step 2 - Load and check

Tweets can contain quote marks, so the files are read with quoting turned off. We check that every row has a label, that no tweet is empty, and that the labels are the three expected ones.

In [ ]:
import pandas as pd, csv

LABELS = ["negative", "neutral", "positive"]          # label ids 0, 1, 2
label2id = {l: i for i, l in enumerate(LABELS)}

def load(split):
    df = pd.read_csv(f"data/raw/sw_{split}.tsv", sep="\t", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    df = df.rename(columns={"ID": "id", "tweet": "text"})
    return df

raw = {s: load(s) for s in FILES}
for s, df in raw.items():
    print(f"{s:5s} rows: {len(df):5d}   labels: {df['label'].value_counts().to_dict()}")
    assert set(df["label"].unique()) <= set(LABELS), "unexpected label value"
    assert (df["text"].str.strip() == "").sum() == 0, "empty tweet found"
print()
raw["train"].head(8)

## Step 3 - Clean the text

The tweets are already stripped of most symbols. We lower-case, remove any web addresses and tidy the spaces. The same function is used later by the web app, so the app sees text in the same form as the models did in training.

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

data = {}
for s, df in raw.items():
    df = df.copy()
    df["text_clean"] = df["text"].map(clean_text)
    df["label_id"] = df["label"].map(label2id)
    data[s] = df

before = sum((df["text_clean"] == "").sum() for df in data.values())
print("Tweets that became empty after cleaning:", before)
for s, df in data.items():
    print(s, "->", len(df), "tweets")

## Step 4 - Class distribution

Sentiment data is usually imbalanced. The neutral class is the biggest. This decides which metric we trust: accuracy would reward a model that always says "neutral", so the main metric is **macro-F1**, which gives all three classes the same weight. The shared task used weighted F1, so we report that too, for comparison.

In [ ]:
import matplotlib.pyplot as plt

tab = pd.DataFrame({s: df["label"].value_counts().reindex(LABELS) for s, df in data.items()})
tab["train %"] = (tab["train"] / tab["train"].sum() * 100).round(1)
print(tab)
print()
print("Share of the majority class (neutral) in test:", round(tab.loc["neutral", "test"] / tab["test"].sum(), 3))

fig, ax = plt.subplots(figsize=(7, 4))
(tab[["train", "val", "test"]]).plot(kind="bar", ax=ax, color=["#12495E", "#6C8E9B", "#C9B98F"])
ax.set_ylabel("Number of tweets"); ax.set_xlabel("")
ax.set_title("Class distribution by split (AfriSenti Swahili)")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("figures/eda/fig_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 5 - Tweet length

Tweets are short, unlike news articles, so the models can use a short maximum sequence length. Look at the examples in Step 7 afterwards: several tweets end abruptly, so some texts in this release appear to be cut off.

In [ ]:
import numpy as np

tr = data["train"]
tr["n_words"] = tr["text_clean"].str.split().str.len()
tr["n_chars"] = tr["text_clean"].str.len()
print(tr[["n_words", "n_chars"]].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
tr["n_chars"].hist(bins=40, ax=axes[0], color="#12495E"); axes[0].set_title("Characters per tweet")
tr["n_words"].hist(bins=40, ax=axes[1], color="#12495E"); axes[1].set_title("Words per tweet")
plt.tight_layout()
plt.savefig("figures/eda/fig_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print()
print(tr.groupby("label")["n_words"].mean().round(1).rename("mean words by label"))

## Step 6 - Vocabulary

How many different words are there, and how many appear only once? Many one-off words mean that a model that treats each word form as a unit sees many unknown words at test time. Swahili adds prefixes and suffixes to words, which increases this.

In [ ]:
from collections import Counter

counter = Counter(w for t in tr["text_clean"] for w in t.split())
total = sum(counter.values())
hapax = sum(1 for c in counter.values() if c == 1)
print("Word tokens:", total)
print("Distinct word forms:", len(counter))
print(f"Words that appear only once: {hapax} ({hapax/len(counter)*100:.1f}% of distinct forms)")
print("Top 25 words:", counter.most_common(25))

# share of validation and test word tokens never seen in training
for s in ["val", "test"]:
    toks = [w for t in data[s]["text_clean"] for w in t.split()]
    unseen = sum(1 for w in toks if w not in counter) / len(toks)
    print(f"{s}: {unseen:.1%} of word tokens do not occur in the training tweets")

## Step 7 - Read some real tweets

Reading the data is part of understanding it. These are random tweets from each class. Notice anything about how label and wording relate? You read Swahili, so your own notes here are useful for the report (for example: names, mixed English and Swahili, questions, hashtags removed).

In [ ]:
pd.set_option("display.max_colwidth", 140)
for lab in LABELS:
    print("=====", lab)
    for t in tr[tr["label"] == lab].sample(5, random_state=3)["text"]:
        print(" -", t[:200])

## Step 8 - Words that separate the classes

For each class we compare how often a word appears in that class with how often it appears elsewhere (a smoothed log-odds ratio). Only words seen at least 5 times are used. This is a quick, rough look, and the baseline model in notebook 2 will learn a similar signal.

In [ ]:
import math

def top_words(label, k=12, min_count=5):
    in_c = Counter(w for t in tr[tr["label"] == label]["text_clean"] for w in t.split())
    out_c = Counter(w for t in tr[tr["label"] != label]["text_clean"] for w in t.split())
    n_in, n_out = sum(in_c.values()), sum(out_c.values())
    scores = {}
    for w, c in in_c.items():
        if c + out_c.get(w, 0) < min_count:
            continue
        scores[w] = math.log((c + 0.5) / (n_in + 1)) - math.log((out_c.get(w, 0) + 0.5) / (n_out + 1))
    return [w for w, _ in sorted(scores.items(), key=lambda kv: -kv[1])[:k]]

for lab in LABELS:
    print(f"{lab:9s}", ", ".join(top_words(lab)))

## Step 9 - Duplicates and overlap between splits

This is the most important check in this notebook. If the same tweet is in the training and the test set, a model can score well just by remembering it. We compare the cleaned text of every tweet.

In [ ]:
def norm(s):
    return s.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

train_texts = set(norm(data["train"]["text_clean"]))
val_texts = set(norm(data["val"]["text_clean"]))

print("Duplicate tweets inside train:", int(norm(data["train"]["text_clean"]).duplicated().sum()))
print("Duplicate tweets inside val  :", int(norm(data["val"]["text_clean"]).duplicated().sum()))
print("Duplicate tweets inside test :", int(norm(data["test"]["text_clean"]).duplicated().sum()))
print()
for s in ["val", "test"]:
    data[s]["seen_in_train"] = norm(data[s]["text_clean"]).isin(train_texts)
    print(f"{s}: {int(data[s]['seen_in_train'].sum())} of {len(data[s])} tweets also appear in train "
          f"({data[s]['seen_in_train'].mean():.1%})")
data["train"]["seen_in_train"] = False
overlap_tv = norm(data["test"]["text_clean"]).isin(train_texts | val_texts).sum()
print("test tweets that appear in train or val:", int(overlap_tv))

# do duplicated texts ever carry different labels?
alltr = data["train"].assign(k=norm(data["train"]["text_clean"]))
conflict = alltr.groupby("k")["label"].nunique()
print("Distinct train texts that appear with more than one label:", int((conflict > 1).sum()))

**How to read this:** if the overlap is only a handful of tweets, it does not change the results. If it were large (as it was for the Kinyarwanda news data), you would report scores both with and without the overlapping tweets. The column `seen_in_train` is saved so that is possible later.

## Step 10 - Save the fixed split

Columns saved: `id`, `text`, `text_clean`, `label` (name), `label_id` (0, 1, 2) and `seen_in_train`. Every later notebook reads these files.

In [ ]:
cols = ["id", "text", "text_clean", "label", "label_id", "seen_in_train"]
for s, df in data.items():
    df[cols].to_csv(f"data/processed/{s}.csv", index=False)
    print(s, len(df), "saved")

import json
meta = {
    "labels": LABELS,
    "sizes": {s: len(df) for s, df in data.items()},
    "class_counts": {s: df["label"].value_counts().reindex(LABELS).astype(int).to_dict() for s, df in data.items()},
    "source": "https://github.com/afrisenti-semeval/afrisent-semeval-2023 (Subtask A, Swahili)",
}
json.dump(meta, open("data/processed/split_info.json", "w"), indent=2)
print(meta)

## Last step - Save your work

Everything this notebook wrote is already in the project folder (`data/` and `figures/`). Save the notebook itself (Cmd+S).

## Done - notebook 1

Paste these outputs back to me (they become the Dataset section of the report):
- Step 2: the row counts and label counts
- Step 4: the class table and the neutral share in test
- Step 5: the length statistics
- Step 6: vocabulary numbers and the unseen-token shares
- Step 7: anything you noticed in the tweets (your own words)
- Step 8: the top words per class
- Step 9: all the duplicate and overlap numbers